In [ ]:
# Standard library imports
import os

# Third-party imports
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Local application imports
from src.data.data_loading import load_volve
from src.data.data_preparation import apply_kalman_filter
from src.evaluation.evaluation import analyze_correlations, plot_series

In [ ]:
if __name__ == "__main__":
    # Configurações iniciais
    well = '15/9-F-12'  # Substitua pelo nome do Well que você está trabalhando
    cum_sum = False
    add_physical_features = False  # Defina como True se quiser adicionar features físicas
    serie_name = 'BORE_OIL_VOL'
    plot_correlated = False

    # Caminho para o arquivo CSV
    file_path = "data/volve/Volve_Equinor.csv"

    # Processa os dados do Well
    df = load_volve(
        data_path=file_path,
        well=well,
        cum_sum=cum_sum,
        add_physical_features=add_physical_features
    )
    
    
    # Análise de correlação
    n_correlated = 12
    top_correlated_features = analyze_correlations(
        df=df,
        series_compared=serie_name,
        top_n=n_correlated  # Por exemplo, as 3 features mais correlacionadas
    )

    print(f"Top {n_correlated} features mais correlacionadas com {serie_name}:", top_correlated_features)
    
    
    top_correlated_features.append(f"{serie_name}_Original")
    top_correlated_features.append(serie_name)
     
    # Aplicar filtro de Kalman nas features mais correlacionadas
    df = apply_kalman_filter(
        df=df,
        features_to_filter=['BORE_OIL_VOL'],
        process_var=1e-6,
        measurement_var=1e-3
    )
    
    if plot_correlated:

        # Plotagem das séries mais correlacionadas
        for feature in top_correlated_features:
            plot_series(
                well=well,
                title=f'BORE_OIL_VOL vs {feature}',
                series=[df['BORE_OIL_VOL'],
                df[feature]],
                series_names=['BORE_OIL_VOL', feature],
                colors=['#206A92', 'yellowgreen'],
                line_styles=['solid', 'dash'],
                x=np.arange(len(df))
            )
    
    # df = df[top_correlated_features]
    # df = df.dropna()
    
    # Exibe as primeiras linhas do DataFrame processado
    print(df.head())

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Optional

# Função para iterar e plotar
def plot_all_columns(df: pd.DataFrame, well: str, colors: Optional[List[str]] = None, line_styles: Optional[List[str]] = None):
    """
    Plota todas as colunas de um DataFrame usando a função plot_series.

    Parameters:
    df (pd.DataFrame): DataFrame contendo os dados a serem plotados.
    well (str): Nome do Well (well).
    colors (Optional[List[str]]): Lista de cores a ser usada em cada coluna.
    line_styles (Optional[List[str]]): Lista de estilos de linha para cada coluna.
    """
    for col in df.columns:
        plot_series(
            well=well,
            title=f"Plot of {col}",
            series=[df[col].values],
            series_names=[col],
            colors=colors,
            line_styles=line_styles,
            x=np.arange(len(df[col]))
        )



# Chamando a função com seu DataFrame e função `plot_series`
plot_all_columns(df, well=well)

In [ ]:
# Standard library imports
import multiprocessing
import os

# Third-party imports
from typing import Any, List, Dict, Callable
import numpy as np
import pandas as pd

# Local application imports
from src.data.data_loading import (
    load_volve,
    load_unisim,
    load_opsd
)
from src.data.data_preparation import (
    filter_data_for_iteration,
    prepare_train_test_sets,
    initialize_prediction_lists,
    calculate_total_iterations,
    organize_wells_by_df_size,
    apply_custom_kalman_filter
)
from src.evaluation.evaluation import evaluate_and_plot_if_needed, compute_metrics, plot_results, evaluate
from src.training.train_utils import fine_tune_and_predict_well
from src.utils.utilities import delete_all_files_in_folder, print_style
from src.training.models_forecast import train_and_evaluate_disruptive

from src.evaluation.evaluation import (
    display_metrics, evaluate_and_plot_results
)

In [ ]:
# Função principal refatorada
def train_and_evaluate_generic(
    forecast_steps: int,
    window_size: int,
    wells: List[str],
    series_name: str,
    total_iterations: int,
    fine_tuning_windows: int,
    generate_synthetic_data: bool = True
) -> None:
    """
    Função genérica para treinamento e avaliação com dados sintéticos.
    
    Parâmetros:
    - forecast_steps (int): Passos de previsão.
    - window_size (int): Tamanho da janela deslizante.
    - wells (List[str]): Lista de nomes de Wells (ou séries).
    - series_name (str): Nome da série de interesse.
    - total_iterations (int): Número total de iterações de controle.
    - fine_tuning_windows (int): Janela de ajuste fino.
    - generate_synthetic_data (bool): Flag para gerar dados sintéticos.
    
    Retorna:
    - None
    """
    # Inicializar listas de teste e predição
    y_test_list = [[] for _ in wells]
    y_pred_list = [[] for _ in wells]
    y_pred_list_filter = [[] for _ in wells]  # Inicializar a lista de filtros

    metrics_accumulator_kalman = []
    metrics_accumulator_no_filter = []

    for control_iteration in range(total_iterations):
        print(f"\nITERAÇÃO: {control_iteration + 1}/{total_iterations}")
        
        # Verifica se é necessário realizar avaliação e plotagem
        if control_iteration % fine_tuning_windows == 0 or control_iteration == (total_iterations - 1):
            if generate_synthetic_data:
                # Gerar dados sintéticos com alguma correlação
                for i in range(len(wells)):
                    # Simular y_test e y_pred com alguma relação
                    y_test = np.random.uniform(50, 150, size=forecast_steps)
                    noise = np.random.normal(0, 10, size=forecast_steps)
                    y_pred = y_test + noise
                    y_test_list[i].extend(y_test)
                    y_pred_list[i].extend(y_pred)
                    
            # Aplicar a filtragem de Kalman
            for well_index in range(len(y_test_list)):
                current_data = y_pred_list[well_index]
                if len(current_data) > 0:
                    filtered_data = apply_custom_kalman_filter(current_data)
                    y_pred_list_filter[well_index] = filtered_data.tolist()
                else:
                    y_pred_list_filter[well_index] = []
            
            if control_iteration == (total_iterations - 1):
                # Última iteração: calcular métricas, plotar e armazenar resultados
                
                # Resultados com Filtro de Kalman
                print_style('Kalman')
                for i in range(len(wells)):
                    well_name = wells[i]
                    y_test = y_test_list[i]
                    y_pred = y_pred_list_filter[i]
                    # Criar objetos TimeSeries
                    # Chamar evaluate_and_plot_results
                    evaluate_and_plot_results(
                        test_series=y_test_list,
                        forecast_series=y_pred_list_filter,
                        well_name=well_name,
                        lag_window=window_size,
                        horizon=forecast_steps,
                        train_cumulative_sum=0.0,  # Assumindo 0 para simplificar
                        sampling_rate=1,           # Assumindo 1 para simplificar
                        metrics_accumulator=metrics_accumulator_kalman,
                        method='Ours',
                        plot_cumulative=False      # Não plotar soma acumulada
                    )
                print("\nMétricas por Well na Última Iteração:")
                display_metrics(metrics_accumulator_kalman)
                
                # Resultados sem Filtro
                print_style('No Filter')
                for i in range(len(wells)):
                    well_name = wells[i]
                    y_test = y_test_list[i]
                    y_pred = y_pred_list[i]
                    # Criar objetos TimeSeries
                    # Chamar evaluate_and_plot_results
                    evaluate_and_plot_results(
                        test_series=y_test_list,
                        forecast_series=y_pred_list,
                        well_name=well_name,
                        lag_window=window_size,
                        horizon=forecast_steps,
                        train_cumulative_sum=0.0,  # Assumindo 0 para simplificar
                        sampling_rate=1,           # Assumindo 1 para simplificar
                        metrics_accumulator=metrics_accumulator_no_filter,
                        method='Ours',
                        plot_cumulative=False      # Não plotar soma acumulada
                    )
                print("\nMétricas por Well na Última Iteração:")
                display_metrics(metrics_accumulator_no_filter)
            else:
                # Iterações intermediárias: apenas plotar resultados
                # Resultados com Filtro de Kalman
                print_style('Kalman')
                for i in range(len(wells)):
                    well_name = wells[i]
                    y_test = y_test_list[i]
                    y_pred = y_pred_list_filter[i]
                    # Criar objetos TimeSeries
                    # Chamar evaluate_and_plot_results
                    evaluate_and_plot_results(
                        test_series=y_test_list,
                        forecast_series=y_pred_list_filter,
                        well_name=well_name,
                        lag_window=window_size,
                        horizon=forecast_steps,
                        train_cumulative_sum=0.0,
                        sampling_rate=1,
                        metrics_accumulator=[],    # Não acumulamos métricas aqui
                        method='Ours',
                        plot_cumulative=False      # Não plotar soma acumulada
                    )
                # Resultados sem Filtro
                print_style('No Filter')
                for i in range(len(wells)):
                    well_name = wells[i]
                    y_test = y_test_list[i]
                    y_pred = y_pred_list[i]
                    # Criar objetos TimeSeries
                    # Chamar evaluate_and_plot_results
                    evaluate_and_plot_results(
                        test_series=y_test_list,
                        forecast_series=y_pred_list,
                        well_name=well_name,
                        lag_window=window_size,
                        horizon=forecast_steps,
                        train_cumulative_sum=0.0,
                        sampling_rate=1,
                        metrics_accumulator=[],
                        method='Ours',
                        plot_cumulative=False      # Não plotar soma acumulada
                    )

# Executando o Pipeline de Teste com Dados Sintéticos

if __name__ == '__main__':
    # Definição dos parâmetros de teste
    forecast_steps = 5
    window_size = 5
    wells = ['Well_A', 'Well_B', 'Well_C', 'Well_D']
    series_name = 'Produção de Óleo'
    total_iterations = 10
    fine_tuning_windows = 2  # Avaliar a cada 2 iterações
    
    # Executar a função de treinamento e avaliação com dados sintéticos
    train_and_evaluate_generic(
        forecast_steps=forecast_steps,
        window_size=window_size,
        wells=wells,
        series_name=series_name,
        total_iterations=total_iterations,
        fine_tuning_windows=fine_tuning_windows,
        generate_synthetic_data=True
    )